# JSAC Migration — Smoke Tests

Incremental tests following the WORKPLAN steps. Each section verifies a step before moving on.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import config_system as CS

## Step 0 — `jsac_metadata` helper

In [ ]:
# Simulate B=2 Blue cars, M_y=2 Yellow + M_g=3 Green each -> K=10
group_ids  = np.array([0,0,0,0,0, 1,1,1,1,1])
link_types = np.array([0,0,1,1,1, 0,0,1,1,1])
M = 5  # links per Blue car

channel_ids, interf_mask, green_mask, yellow_mask = CS.jsac_metadata(group_ids, link_types, M)

print('channel_ids:', channel_ids)
print('interf_mask (non-zero):', int(interf_mask.sum()))
print('interf_mask:\n', interf_mask.astype(int))
print('green_mask:', green_mask)
print('yellow_mask:', yellow_mask)
print('n_green:', green_mask.sum(), 'n_yellow:', yellow_mask.sum())

## Step 1
### Step 1A — `init_parameters_JSAC`

In [ ]:
p = CS.init_parameters_JSAC(n_blue=5, n_yellow_per_blue=2, n_green_per_blue=3)
print('n_links:', p.n_links)            # 25
print('n_blue:', p.n_blue)
print('setting_str:', p.setting_str)
print('tx_power:', p.tx_power, 'W')
print('min_blue_dist:', p.min_blue_dist, 'm')
print('rx_min_radius:', p.rx_min_radius, 'm')
print('rx_max_radius:', p.rx_max_radius, 'm')
print('min_rx_separation:', p.min_rx_separation, 'm')

### Step 1B — `layout_generate_jsac` & `generate_layouts_jsac`

In [ ]:
np.random.seed(42)
p = CS.init_parameters_JSAC(n_blue=5, n_yellow_per_blue=2, n_green_per_blue=3)
layout, dists, group_ids, link_types = CS.layout_generate_jsac(p)

K = p.n_links
M = p.n_yellow_per_blue + p.n_green_per_blue
print('K:', K, '  layout shape:', layout.shape, '  dists shape:', dists.shape)
print('group_ids:', group_ids)
print('link_types:', link_types)

# Verify: links in same group share identical Tx coordinates
for b in range(p.n_blue):
    idxs = np.where(group_ids == b)[0]
    txs = layout[idxs, :2]
    assert np.allclose(txs, txs[0]), f'Group {b} Tx coords not identical!'
print('OK: All links in same group share identical Tx coords')

# Verify: diagonal distances are in [rx_min, rx_max]
diag_dists = np.diag(dists)
print(f'Direct-link dists: min={diag_dists.min():.2f}, max={diag_dists.max():.2f}'
      f' (expected [{p.rx_min_radius}, {p.rx_max_radius}])')

# Verify: min Rx separation within each cluster
for b in range(p.n_blue):
    idxs = np.where(group_ids == b)[0]
    rxs = layout[idxs, 2:]
    for i in range(len(idxs)):
        for j in range(i+1, len(idxs)):
            d = np.linalg.norm(rxs[i] - rxs[j])
            assert d >= p.min_rx_separation - 1e-9, \
                f'Cluster {b}: Rx {i},{j} too close ({d:.3f}m)'
print(f'OK: All intra-cluster Rx separations >= {p.min_rx_separation}m')

# Verify: Blue cars are >= min_blue_dist apart
blue_coords = np.array([layout[b * M, :2] for b in range(p.n_blue)])
for i in range(p.n_blue):
    for j in range(i+1, p.n_blue):
        d = np.linalg.norm(blue_coords[i] - blue_coords[j])
        assert d >= p.min_blue_dist - 1e-9, f'Blue {i},{j} too close ({d:.2f}m)'
print(f'OK: All Blue cars >= {p.min_blue_dist}m apart')

### Layout visualization

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))

for b in range(p.n_blue):
    idxs = np.where(group_ids == b)[0]
    bx, by = layout[idxs[0], 0], layout[idxs[0], 1]

    # Blue car (Tx)
    ax.plot(bx, by, 's', color='blue', markersize=10, zorder=5,
            label='Blue (Tx)' if b == 0 else None)

    for idx in idxs:
        rx_x, rx_y = layout[idx, 2], layout[idx, 3]
        if link_types[idx] == 0:  # Yellow
            color, marker = 'goldenrod', '^'
            label = 'Yellow (Sensing Rx)' if b == 0 and idx == idxs[0] else None
        else:  # Green
            color, marker = 'green', 'o'
            label = 'Green (Comm Rx)' if b == 0 and idx == idxs[p.n_yellow_per_blue] else None
        ax.plot(rx_x, rx_y, marker, color=color, markersize=7, zorder=4, label=label)
        ax.plot([bx, rx_x], [by, rx_y], '-', color=color, alpha=0.4, linewidth=1)

ax.set_xlim(0, p.field_length)
ax.set_ylim(0, p.field_length)
ax.set_aspect('equal')
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title(f'JSAC Layout: B={p.n_blue}, M_y={p.n_yellow_per_blue}, M_g={p.n_green_per_blue}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Step 1C — `compute_channel_losses_jsac` (orthogonal channel masking)

In [ ]:
np.random.seed(42)
p = CS.init_parameters_JSAC(n_blue=3, n_yellow_per_blue=2, n_green_per_blue=3)
K = p.n_links  # 15
M = p.n_yellow_per_blue + p.n_green_per_blue  # 5

layout, dists, group_ids, link_types = CS.layout_generate_jsac(p)
channel_ids, interf_mask, green_mask, yellow_mask = CS.jsac_metadata(group_ids, link_types, M)

print('channel_ids:', channel_ids)
print(f'interf_mask: {int(interf_mask.sum())} non-zero (expected {M * p.n_blue * (p.n_blue-1)})')

dists_batch = dists[np.newaxis, :, :]
path_losses, channel_losses = CS.compute_channel_losses_jsac(p, dists_batch, interf_mask)
cl = channel_losses[0]

# Check 1: diagonal (direct links) > 0
diag = np.diag(cl)
assert np.all(diag > 0), 'FAIL: some diagonal entries are zero!'
print(f'\nOK: All {K} direct links > 0')

# Check 2: interfering entries (same channel, different group) > 0
interf_values = cl[interf_mask.astype(bool)]
assert np.all(interf_values > 0), 'FAIL: some interfering entries are zero!'
print(f'OK: All {len(interf_values)} interfering entries > 0')

# Check 3: all other off-diagonal entries == 0
non_interf = (1 - interf_mask - np.eye(K)).astype(bool)
non_interf_values = cl[non_interf]
assert np.all(non_interf_values == 0), 'FAIL: non-interfering entries are non-zero!'
print(f'OK: All {non_interf.sum()} non-interfering off-diagonal entries are zero')

print(f'\nChannel matrix sparsity: {(cl > 0).sum()} / {K*K} entries non-zero')

## Step 2 — `utils_objective.py`: Green sum-rate & Yellow SINR violation

In [ ]:
import utils_objective as UO

np.random.seed(42)
p = CS.init_parameters_JSAC(n_blue=3, n_yellow_per_blue=2, n_green_per_blue=3)
K = p.n_links  # 15
M = p.n_links_per_blue  # 5

# Generate 50 layouts for statistical stability
layouts, dists_batch, group_ids, link_types = CS.generate_layouts_jsac(p, 50)
channel_ids, interf_mask, green_mask, yellow_mask = CS.jsac_metadata(group_ids, link_types, M)
path_losses, channel_losses = CS.compute_channel_losses_jsac(p, dists_batch, interf_mask)

dLoss = UO.get_directLink_channel_losses(channel_losses)
cLoss = UO.get_crossLink_channel_losses(channel_losses)

# Equal power allocation (normalized: each group sums to 1)
allocs = np.ones((50, K)) * (1.0 / M)

# 2B: Green sum-rate
green_sr = UO.compute_green_sumrate(p, allocs, dLoss, cLoss, green_mask)
total_rates = UO.compute_rates(p, allocs, dLoss, cLoss)
total_sr = np.mean(np.sum(total_rates, axis=1))
print(f'Green sum-rate: {green_sr:.4f} bits/s/Hz')
print(f'Total sum-rate: {total_sr:.4f} bits/s/Hz')
assert green_sr < total_sr, 'Green sum-rate should be < total!'
assert green_sr > 0, 'Green sum-rate should be positive!'
print('OK: 0 < Green sum-rate < Total sum-rate')

# 2C: Yellow SINR violation
sinr_min = 1.0  # 0 dB threshold
viol_rate, yellow_sinrs = UO.compute_yellow_sinr_violation(p, allocs, dLoss, cLoss, yellow_mask, sinr_min)
print(f'\nYellow SINR violation rate (>= 0 dB): {viol_rate:.1f}%')
print(f'Yellow SINR stats: min={yellow_sinrs.min():.2f}, '
      f'median={np.median(yellow_sinrs):.2f}, max={yellow_sinrs.max():.2f}')
assert 0 <= viol_rate <= 100
assert yellow_sinrs.shape == (50, yellow_mask.sum())
print(f'OK: yellow_sinrs shape = {yellow_sinrs.shape}')

print('\n=== Step 2 smoke tests passed ===')

## Step 3 — `baselines.py`: WMMSE_JSAC & Naive Equal-Power

Verify:
- `naive_equal_power`: per-group sums = 1, plus rate/SINR metrics
- `batch_WMMSE2_JSAC`: per-group power ≤ Pmax, Yellow SINR constraint, Green sum-rate

In [ ]:
import baselines as BL

np.random.seed(42)
para = CS.init_parameters_JSAC(n_blue=10, n_yellow_per_blue=2, n_green_per_blue=3, field_length=225)
K = para.n_links  # 25
M = para.n_links_per_blue  # 5
var = para.output_noise_power / para.tx_power
num_layouts = 50
SHUFFLE_CHANNELS = False

# Generate layouts + channels
layouts, dists, group_ids, link_types = CS.generate_layouts_jsac(para, num_layouts)
channel_ids, interf_mask, green_mask, yellow_mask = CS.jsac_metadata(group_ids, link_types, M, shuffle_channels=SHUFFLE_CHANNELS)
path_losses, channel_losses = CS.compute_channel_losses_jsac(para, dists, interf_mask)

dLoss = UO.get_directLink_channel_losses(channel_losses)
cLoss = UO.get_crossLink_channel_losses(channel_losses)

H = np.sqrt(channel_losses)
Pmax_norm = 1.0
sinr_min_dB = 0 # 0 dB
sinr_min = 10 ** (sinr_min_dB / 10)  # 1.0 linear

# =====================================================================
# Naive Equal-Power
# =====================================================================
allocs_eq = BL.naive_equal_power(para, group_ids)

print('=== Naive Equal-Power ===')
print(f'allocs shape: {allocs_eq.shape}, values: {allocs_eq[:M]}')

# Check per-group sums = 1
for b in range(para.n_blue):
    links = np.where(group_ids == b)[0]
    assert np.isclose(np.sum(allocs_eq[links]), 1.0), f'Group {b} sum != 1'
print('OK: All per-group sums = 1.0')

# Broadcast to batch for evaluation
allocs_eq_batch = np.tile(allocs_eq, (num_layouts, 1))

# Green sum-rate
green_sr_eq = UO.compute_green_sumrate(para, allocs_eq_batch, dLoss, cLoss, green_mask)
total_rates_eq = UO.compute_rates(para, allocs_eq_batch, dLoss, cLoss)
total_sr_eq = np.mean(np.sum(total_rates_eq, axis=1))
print(f'Green sum-rate: {green_sr_eq:.4f}')
print(f'Total sum-rate: {total_sr_eq:.4f}')

# Yellow SINR violation
viol_eq, yellow_sinrs_eq = UO.compute_yellow_sinr_violation(
    para, allocs_eq_batch, dLoss, cLoss, yellow_mask, sinr_min)
print(f'Yellow SINR violation rate (>= {sinr_min_dB} dB): {viol_eq:.1f}%')
print(f'Yellow SINR: min={yellow_sinrs_eq.min():.2f}, '
      f'median={np.median(yellow_sinrs_eq):.2f}, mean={np.mean(yellow_sinrs_eq):.2f}')

# =====================================================================
# WMMSE_JSAC
# =====================================================================
alpha = np.ones((num_layouts, K))
alpha[:, yellow_mask] = 0.1

p_opt = BL.batch_WMMSE2_JSAC(num_layouts, alpha, H, Pmax_norm, var,
                              group_ids, sinr_min, yellow_mask)

print('\n=== WMMSE_JSAC ===')
print(f'p_opt shape: {p_opt.shape}')

# Verify per-group power <= Pmax
all_ok = True
for b in range(para.n_blue):
    links = np.where(group_ids == b)[0]
    group_powers = np.sum(p_opt[:, links], axis=1)
    max_gp = np.max(group_powers)
    if max_gp > Pmax_norm + 1e-6:
        print(f'  Group {b}: max={max_gp:.6f} VIOLATION!')
        all_ok = False
    else:
        print(f'  Group {b}: max_power={max_gp:.6f}, mean_power={np.mean(group_powers):.6f}')
assert all_ok, 'Per-group power constraint violated!'
print('OK: All groups respect Pmax')

# Green sum-rate
green_sr_wm = UO.compute_green_sumrate(para, p_opt, dLoss, cLoss, green_mask)
total_rates_wm = UO.compute_rates(para, p_opt, dLoss, cLoss)
total_sr_wm = np.mean(np.sum(total_rates_wm, axis=1))
print(f'Green sum-rate: {green_sr_wm:.4f}')
print(f'Total sum-rate: {total_sr_wm:.4f}')

# Yellow SINR violation
viol_wm, yellow_sinrs_wm = UO.compute_yellow_sinr_violation(
    para, p_opt, dLoss, cLoss, yellow_mask, sinr_min)
print(f'Yellow SINR violation rate (>= {sinr_min_dB} dB): {viol_wm:.1f}%')
print(f'Yellow SINR: min={yellow_sinrs_wm.min():.2f}, '
      f'median={np.median(yellow_sinrs_wm):.2f}, mean={np.mean(yellow_sinrs_wm):.2f}')

# =====================================================================
# Comparison summary
# =====================================================================
print('\n=== Comparison ===')
print(f'{"Metric":<30} {"Equal-Power":>12} {"WMMSE_JSAC":>12}')
print('-' * 56)
print(f'{"Green sum-rate":<30} {green_sr_eq:>12.4f} {green_sr_wm:>12.4f}')
print(f'{"Total sum-rate":<30} {total_sr_eq:>12.4f} {total_sr_wm:>12.4f}')
print(f'{"Yellow SINR violation %":<30} {viol_eq:>11.1f}% {viol_wm:>11.1f}%')

assert green_sr_wm > green_sr_eq, 'WMMSE should beat equal-power on Green sum-rate!'
print('\nOK: WMMSE_JSAC Green sum-rate > Equal-Power (as expected)')
print('=== Step 3 smoke tests passed ===')

## Step 4 — `model_gnn.py`: Graph Construction, Training & Evaluation

Note: It's using old `WirelessScaler` so cannot run with modified new codes.

Verify:
- `build_graph()` uses `interf_mask` + intra-group edges; node feat: [dist, loss, link_type, power] (4D), edge feat: [dist, type, channel_gain] (3D)
- `IGCNet` outputs raw logits (no Sigmoid); 4 message-passing iterations; per-group softmax gives power sums = 1.0
- `constrained_sumrate_loss_jsac()`: Green-only objective + Yellow SINR penalty
- `train_epoch` / `eval_epoch` work end-to-end
- GNN performance: Equal Power < GNN ≈ WMMSE on Green sum-rate

In [ ]:
# import torch
# from torch_geometric.loader import DataLoader
# import model_gnn as MG
# from main_supporter import WirelessScaler

# np.random.seed(42)
# torch.manual_seed(42)

# para = CS.init_parameters_JSAC(n_blue=10, n_yellow_per_blue=2, n_green_per_blue=3, field_length=225)
# K = para.n_links  # 50
# M = para.n_links_per_blue  # 5
# n_blue = para.n_blue
# var = para.output_noise_power / para.tx_power
# sinr_min = 1.0  # 0 dB
# penalty_weight = 2.0
# TRAINING_EPOCHS = 30

# # =====================================================================
# # 4A/4B: Graph construction with interf_mask + intra-group edges
# # =====================================================================
# num_train = 5000
# layouts, dists, group_ids, link_types = CS.generate_layouts_jsac(para, num_train)
# channel_ids, interf_mask, green_mask, yellow_mask = CS.jsac_metadata(group_ids, link_types, M)
# _, channel_losses = CS.compute_channel_losses_jsac(para, dists, interf_mask)

# ds = WirelessScaler(); ds.fit(1.0 / dists)
# ls = WirelessScaler(); ls.fit(np.sqrt(channel_losses))
# nd = ds.transform(1.0 / dists)
# nl = ls.transform(np.sqrt(channel_losses))

# train_data = MG.proc_data(channel_losses, dists, nd, nl, K, interf_mask, group_ids, green_mask, yellow_mask)
# train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# # Verify graph structure
# d0 = train_data[0]
# n_interf = int(interf_mask.sum())
# n_intra = int(sum(np.sum(group_ids == b) * (np.sum(group_ids == b) - 1) for b in range(n_blue)))
# print(f'Graph 0: x={d0.x.shape} (4 node feat), edge_attr={d0.edge_attr.shape} (3 edge feat)')
# print(f'  Edges: {d0.edge_index.shape[1]} = {n_interf} interf + {n_intra} intra-group')
# print(f'  green={d0.green_mask.sum()}, yellow={d0.yellow_mask.sum()}')
# assert d0.x.shape == (K, 4), 'Expected 4 node features'
# assert d0.edge_attr.shape[1] == 3, 'Expected 3 edge features'
# assert d0.edge_index.shape[1] == n_interf + n_intra, 'Edge count mismatch!'
# print('OK: Graph structure verified')

# # =====================================================================
# # 4C: Per-group softmax — verify power sums = 1.0
# # =====================================================================
# model = MG.IGCNet()
# loader_check = DataLoader(train_data[:4], batch_size=4, shuffle=False)
# for batch in loader_check:
#     out = model(batch)
#     raw_p = out[:, 3]
#     p = MG.apply_group_softmax(raw_p, batch.group_ids, batch.batch, n_blue)
#     p_mat = p.reshape(batch.num_graphs, K)
#     for g in range(batch.num_graphs):
#         for b in range(n_blue):
#             glinks = (group_ids == b)
#             group_sum = p_mat[g, glinks].sum().item()
#             assert abs(group_sum - 1.0) < 1e-5, f'Graph {g}, Blue {b}: sum={group_sum}'
#     print(f'OK: All per-group power sums = 1.0 (checked {batch.num_graphs} graphs x {n_blue} groups)')
#     break

# # =====================================================================
# # 4D/4E: Train GNN
# # =====================================================================
# num_test = 500
# layouts_t, dists_t, _, _ = CS.generate_layouts_jsac(para, num_test)
# _, channel_losses_t = CS.compute_channel_losses_jsac(para, dists_t, interf_mask)
# nd_t = ds.transform(1.0 / dists_t)
# nl_t = ls.transform(np.sqrt(channel_losses_t))
# test_data = MG.proc_data(channel_losses_t, dists_t, nd_t, nl_t, K, interf_mask, group_ids, green_mask, yellow_mask)
# test_loader = DataLoader(test_data, batch_size=64, shuffle=False)
# dLoss_t = UO.get_directLink_channel_losses(channel_losses_t)
# cLoss_t = UO.get_crossLink_channel_losses(channel_losses_t)

# device = torch.device('cpu')
# model = MG.IGCNet().to(device)
# optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

# print(f'\nTraining GNN ({TRAINING_EPOCHS} epochs)...')
# for epoch in range(TRAINING_EPOCHS):
#     train_loss = MG.train_epoch(model, train_loader, optimizer, device, K, var, n_blue, sinr_min, penalty_weight)
#     scheduler.step()
#     if (epoch + 1) % 5 == 0:
#         _, green_sr, viol_rate = MG.eval_epoch(
#             model, test_loader, device, K, para, dLoss_t, cLoss_t,
#             green_mask, yellow_mask, n_blue, sinr_min, penalty_weight)
#         print(f'  Epoch {epoch+1:2d}: loss={train_loss:.4f}, green_SR={green_sr:.2f}, viol={viol_rate:.1f}%')

# # =====================================================================
# # Comparison: Equal Power vs GNN vs WMMSE
# # =====================================================================
# allocs_eq = BL.naive_equal_power(para, group_ids)
# allocs_eq_batch = np.tile(allocs_eq, (num_test, 1))
# green_sr_eq = UO.compute_green_sumrate(para, allocs_eq_batch, dLoss_t, cLoss_t, green_mask)
# viol_eq, _ = UO.compute_yellow_sinr_violation(para, allocs_eq_batch, dLoss_t, cLoss_t, yellow_mask, sinr_min)

# H_test = np.sqrt(channel_losses_t)
# alpha = np.ones((num_test, K))
# alpha[:, yellow_mask] = 0.1
# p_wm = BL.batch_WMMSE2_JSAC(num_test, alpha, H_test, 1.0, var, group_ids, sinr_min, yellow_mask)
# green_sr_wm = UO.compute_green_sumrate(para, p_wm, dLoss_t, cLoss_t, green_mask)
# viol_wm, _ = UO.compute_yellow_sinr_violation(para, p_wm, dLoss_t, cLoss_t, yellow_mask, sinr_min)

# _, green_sr_gnn, viol_gnn = MG.eval_epoch(
#     model, test_loader, device, K, para, dLoss_t, cLoss_t,
#     green_mask, yellow_mask, n_blue, sinr_min, penalty_weight)

# print('\n' + '='*56)
# print(f'{"Method":<20} {"Green SR":>10} {"Violation %":>12}')
# print('-' * 56)
# print(f'{"Equal Power":<20} {green_sr_eq:>10.2f} {viol_eq:>11.1f}%')
# print(f'{"GNN (Ours)":<20} {green_sr_gnn:>10.2f} {viol_gnn:>11.1f}%')
# print(f'{"WMMSE_JSAC":<20} {green_sr_wm:>10.2f} {viol_wm:>11.1f}%')
# print('='*56)
# print(f'GNN/WMMSE ratio: {green_sr_gnn/green_sr_wm*100:.1f}%')

# assert green_sr_gnn > 0, 'GNN Green sum-rate should be positive'
# assert 0 <= viol_gnn <= 100, 'Violation rate should be in [0, 100]'
# # GNN should significantly beat equal power and be close to WMMSE
# assert green_sr_gnn > green_sr_eq * 1.05, 'GNN should clearly beat equal power (>5% margin)'
# assert green_sr_gnn > green_sr_wm * 0.90, 'GNN should reach at least 90% of WMMSE'
# print('\n=== Step 4 smoke tests passed ===')

## Step 5 — `main_supporter.py`: Scaler, Data Generation & Evaluation Helpers

Verify:
- `WirelessScaler` uses 3-category normalization (direct-sensing, direct-comm, interference)
- `generate_training_dataset()` encapsulates layout gen + channel + scaler fitting
- `generate_evaluation_dataset()` uses pre-fitted scalers from training
- GNN trained with new helpers matches Step 4 performance (Green SR, violation rate)
- Power usage visualization: Yellow vs Green per method

In [ ]:
import torch
from torch_geometric.loader import DataLoader
import model_gnn as MG
import main_supporter as Sapo

np.random.seed(42)
torch.manual_seed(42)

para = CS.init_parameters_JSAC(n_blue=10, n_yellow_per_blue=2, n_green_per_blue=3, field_length=225)
K = para.n_links  # 50
M = para.n_links_per_blue  # 5
n_blue = para.n_blue
var = para.output_noise_power / para.tx_power
sinr_min = 1.0  # 0 dB
penalty_weight = 2.0
TRAINING_EPOCHS = 30

# =====================================================================
# 5A: generate_training_dataset (encapsulates layout+channel+scaler)
# =====================================================================
train_data, ds, ls, meta = Sapo.generate_training_dataset(para, 5000)
green_mask = meta['green_mask']
yellow_mask = meta['yellow_mask']
group_ids = meta['group_ids']
interf_mask = meta['interf_mask']

print('=== 5A: WirelessScaler 3-category statistics ===')
print(f'  dist_scaler:')
print(f'    sense  mean={ds.sense_mean:.4f}  std={ds.sense_var:.4f}')
print(f'    comm   mean={ds.comm_mean:.4f}  std={ds.comm_var:.4f}')
print(f'    interf mean={ds.interf_mean:.4f}  std={ds.interf_var:.4f}')
print(f'  loss_scaler:')
print(f'    sense  mean={ls.sense_mean:.6f}  std={ls.sense_var:.6f}')
print(f'    comm   mean={ls.comm_mean:.6f}  std={ls.comm_var:.6f}')
print(f'    interf mean={ls.interf_mean:.6f}  std={ls.interf_var:.6f}')

# Verify: sense ≈ comm (same channel model), interf << direct (much weaker)
assert abs(ds.sense_mean - ds.comm_mean) / ds.sense_mean < 0.1, 'sense/comm dist stats should be similar'
assert ds.interf_mean < ds.sense_mean * 0.2, 'interference dist should be much smaller than direct'
print('OK: 3-category stats verified (sense ≈ comm, interf << direct)')

# =====================================================================
# 5B: generate_evaluation_dataset (uses pre-fitted scalers)
# =====================================================================
print('\n=== 5B: generate_evaluation_dataset ===')
num_test = 500
ch_test, test_data, dLoss_t, cLoss_t, _, conf = Sapo.generate_evaluation_dataset(
    para, num_test, ds, ls, interf_mask, group_ids, green_mask, yellow_mask
)
print(f'  ch_test: {ch_test.shape}, test_data: {len(test_data)} graphs')
print(f'  dLoss: {dLoss_t.shape}, cLoss: {cLoss_t.shape}')
assert ch_test.shape == (num_test, K, K)
assert len(test_data) == num_test
print('OK: Evaluation dataset generated with pre-fitted scalers')

# =====================================================================
# 5C: Train GNN using the new helpers (repeat Step 4 comparison)
# =====================================================================
print('\n=== 5C: Train + Eval GNN ===')
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

device = torch.device('cpu')
model = MG.IGCNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

for epoch in range(TRAINING_EPOCHS):
    train_loss = MG.train_epoch(model, train_loader, optimizer, device, K, var, n_blue, sinr_min, penalty_weight)
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        _, green_sr, viol_rate = MG.eval_epoch(
            model, test_loader, device, K, para, dLoss_t, cLoss_t,
            green_mask, yellow_mask, n_blue, sinr_min, penalty_weight)
        print(f'  Epoch {epoch+1:2d}: loss={train_loss:.4f}, green_SR={green_sr:.2f}, viol={viol_rate:.1f}%')

# =====================================================================
# Comparison: Equal Power vs GNN vs WMMSE
# =====================================================================
allocs_eq = BL.naive_equal_power(para, group_ids)
allocs_eq_batch = np.tile(allocs_eq, (num_test, 1))
green_sr_eq = UO.compute_green_sumrate(para, allocs_eq_batch, dLoss_t, cLoss_t, green_mask)
viol_eq, _ = UO.compute_yellow_sinr_violation(para, allocs_eq_batch, dLoss_t, cLoss_t, yellow_mask, sinr_min)

H_test = np.sqrt(ch_test)
alpha = np.ones((num_test, K))
alpha[:, yellow_mask] = 0.1
p_wm = BL.batch_WMMSE2_JSAC(num_test, alpha, H_test, 1.0, var, group_ids, sinr_min, yellow_mask)
green_sr_wm = UO.compute_green_sumrate(para, p_wm, dLoss_t, cLoss_t, green_mask)
viol_wm, _ = UO.compute_yellow_sinr_violation(para, p_wm, dLoss_t, cLoss_t, yellow_mask, sinr_min)

_, green_sr_gnn, viol_gnn = MG.eval_epoch(
    model, test_loader, device, K, para, dLoss_t, cLoss_t,
    green_mask, yellow_mask, n_blue, sinr_min, penalty_weight)

print('\n' + '='*56)
print(f'{"Method":<20} {"Green SR":>10} {"Violation %":>12}')
print('-' * 56)
print(f'{"Equal Power":<20} {green_sr_eq:>10.2f} {viol_eq:>11.1f}%')
print(f'{"GNN (Ours)":<20} {green_sr_gnn:>10.2f} {viol_gnn:>11.1f}%')
print(f'{"WMMSE_JSAC":<20} {green_sr_wm:>10.2f} {viol_wm:>11.1f}%')
print('='*56)
print(f'GNN/WMMSE ratio: {green_sr_gnn/green_sr_wm*100:.1f}%')

# =====================================================================
# Power Usage Analysis: Yellow vs Green per method
# =====================================================================
print('\n=== Power Usage Analysis ===')

# Extract GNN power allocations
model.eval()
all_powers_gnn = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        raw_p = out[:, 3]
        p = MG.apply_group_softmax(raw_p, data.group_ids, data.batch, n_blue)
        all_powers_gnn.append(p.reshape(data.num_graphs, K).cpu().numpy())
all_powers_gnn = np.concatenate(all_powers_gnn, axis=0)

# Table: avg power per link type
print(f'\n{"Method":<15} {"Yellow avg":>10} {"Yellow std":>10} {"Green avg":>10} {"Green std":>10} {"Group total":>12}')
print('-' * 70)
for name, allocs in [('Equal Power', allocs_eq_batch), ('WMMSE', p_wm), ('GNN', all_powers_gnn)]:
    yp = allocs[:, yellow_mask]
    gp = allocs[:, green_mask]
    group_tot = np.mean([allocs[:, group_ids == b].sum(axis=1).mean() for b in range(n_blue)])
    print(f'{name:<15} {yp.mean():>10.4f} {yp.std():>10.4f} {gp.mean():>10.4f} {gp.std():>10.4f} {group_tot:>12.4f}')

# Bar chart: power distribution per method, Yellow vs Green
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
methods = [('Equal Power', allocs_eq_batch), ('WMMSE', p_wm), ('GNN', all_powers_gnn)]

for ax, (name, allocs) in zip(axes, methods):
    # Average across all test layouts
    avg_power = allocs.mean(axis=0)  # (K,)
    x = np.arange(K)
    colors = ['#FFD700' if yellow_mask[i] else '#2ECC71' for i in range(K)]
    ax.bar(x, avg_power, color=colors, width=0.8, edgecolor='none')
    ax.set_title(name)
    ax.set_xlabel('Link index')
    if ax == axes[0]:
        ax.set_ylabel('Avg power allocation')
    ax.set_xlim(-0.5, K - 0.5)
    # Add group separators
    for b in range(1, n_blue):
        ax.axvline(x=b * M - 0.5, color='gray', linestyle=':', alpha=0.5)

# Legend
from matplotlib.patches import Patch
axes[-1].legend(handles=[
    Patch(facecolor='#FFD700', label='Yellow (sensing)'),
    Patch(facecolor='#2ECC71', label='Green (comm)')
], loc='upper right')
fig.suptitle('Power Allocation per Link (averaged over test layouts)', y=1.02)
plt.tight_layout()
plt.show()

# Box plot: per-group Yellow vs Green power share
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, (name, allocs) in zip(axes, methods):
    yellow_shares = []
    green_shares = []
    for b in range(n_blue):
        mask_b = (group_ids == b)
        group_total = allocs[:, mask_b].sum(axis=1, keepdims=True)  # (N, 1)
        y_in_group = yellow_mask[mask_b]
        g_in_group = green_mask[mask_b]
        yellow_shares.append(allocs[:, mask_b][:, y_in_group].sum(axis=1) / (group_total.squeeze() + 1e-12))
        green_shares.append(allocs[:, mask_b][:, g_in_group].sum(axis=1) / (group_total.squeeze() + 1e-12))
    # Flatten across groups
    yellow_shares = np.concatenate(yellow_shares)
    green_shares = np.concatenate(green_shares)

    bp = ax.boxplot([yellow_shares, green_shares], labels=['Yellow', 'Green'],
                    patch_artist=True, widths=0.5)
    bp['boxes'][0].set_facecolor('#FFD700')
    bp['boxes'][1].set_facecolor('#2ECC71')
    ax.set_title(name)
    ax.set_ylabel('Share of group power' if ax == axes[0] else '')
    ax.set_ylim(-0.05, 1.05)
    # Reference line: equal share
    n_y = para.n_yellow_per_blue
    n_g = para.n_green_per_blue
    ax.axhline(y=n_y / M, color='#FFD700', linestyle='--', alpha=0.5, label=f'Equal ({n_y}/{M})')
    ax.axhline(y=n_g / M, color='#2ECC71', linestyle='--', alpha=0.5, label=f'Equal ({n_g}/{M})')
    ax.legend(fontsize=8)

fig.suptitle('Power Share within Each Blue Car Group (Yellow vs Green)', y=1.02)
plt.tight_layout()
plt.show()

# =====================================================================
# Assertions
# =====================================================================
assert green_sr_gnn > green_sr_eq * 1.05, 'GNN should clearly beat equal power (>5% margin)'
assert green_sr_gnn > green_sr_wm * 0.90, 'GNN should reach at least 90% of WMMSE'
print('\n=== Step 5 smoke tests passed ===')

## Step 7 — `test_JSAC.py`: Single-Scenario Deep Evaluation

Verify that `test_JSAC.py` functions work correctly:
- `calculate_metrics_jsac()` returns all expected metrics
- Plots render without errors
- GNN beats Equal Power, approaches WMMSE

In [ ]:
import test_JSAC

# Reuse training artifacts from Step 5 (model, scalers, test data)
# to verify calculate_metrics_jsac and plotting functions

# =====================================================================
# 7A: calculate_metrics_jsac — verify all metric fields
# =====================================================================
print('=== 7A: calculate_metrics_jsac ===')

# Equal Power
allocs_eq = np.tile(BL.naive_equal_power(para, group_ids), (num_test, 1))
m_eq = test_JSAC.calculate_metrics_jsac(
    para, allocs_eq, dLoss_t, cLoss_t,
    green_mask, yellow_mask, group_ids, sinr_min)

# WMMSE
alpha_w = np.ones((num_test, K))
alpha_w[:, yellow_mask] = 0.1
H_t = np.sqrt(ch_test)
allocs_wm = BL.batch_WMMSE2_JSAC(
    num_test, alpha_w, H_t, 1.0, var, group_ids, sinr_min, yellow_mask)
m_wm = test_JSAC.calculate_metrics_jsac(
    para, allocs_wm, dLoss_t, cLoss_t,
    green_mask, yellow_mask, group_ids, sinr_min)

# GNN
model.eval()
gnn_pows = []
with torch.no_grad():
    for data in test_loader:
        data = data.to(device)
        out = model(data)
        raw_p = out[:, 3]
        p = MG.apply_group_softmax(raw_p, data.group_ids, data.batch, n_blue)
        gnn_pows.append(p.reshape(data.num_graphs, K).cpu().numpy())
allocs_gnn = np.concatenate(gnn_pows, axis=0)
m_gnn = test_JSAC.calculate_metrics_jsac(
    para, allocs_gnn, dLoss_t, cLoss_t,
    green_mask, yellow_mask, group_ids, sinr_min)

# Verify all expected keys exist
expected_keys = ['green_sumrate', 'yellow_viol_rate', 'green_rates',
                 'yellow_sinrs', 'all_rates', 'all_sinrs',
                 'group_power_util', 'jains_green', 'percentiles']
for key in expected_keys:
    assert key in m_eq, f'Missing key: {key}'
print(f'  All {len(expected_keys)} metric keys present')

# Shape checks
n_green = green_mask.sum()
n_yellow = yellow_mask.sum()
assert m_eq['green_rates'].shape == (num_test, n_green)
assert m_eq['yellow_sinrs'].shape == (num_test, n_yellow)
assert m_eq['group_power_util'].shape == (num_test, n_blue)
print(f'  Shapes OK: green_rates={m_eq["green_rates"].shape}, '
      f'yellow_sinrs={m_eq["yellow_sinrs"].shape}')

# Sanity: GNN > Equal Power, WMMSE >= GNN
print(f'  Green SR: EqPow={m_eq["green_sumrate"]:.2f}, '
      f'GNN={m_gnn["green_sumrate"]:.2f}, '
      f'WMMSE={m_wm["green_sumrate"]:.2f}')
assert m_gnn['green_sumrate'] > m_eq['green_sumrate'] * 1.05
print('OK: GNN beats Equal Power by >5%')

# =====================================================================
# 7B: print_metrics_table — verify it prints without error
# =====================================================================
print('\n=== 7B: print_metrics_table ===')
all_metrics = {'Equal Power': m_eq, 'WMMSE': m_wm, 'GNN': m_gnn}
test_JSAC.print_metrics_table(all_metrics, 0.0)

# =====================================================================
# 7C: Dashboard plot
# =====================================================================
print('=== 7C: Dashboard plot ===')
allocs_dict = {'Equal Power': allocs_eq, 'WMMSE': allocs_wm, 'GNN': allocs_gnn}
test_JSAC.plot_dashboard(
    all_metrics, sinr_min, 0.0, para, group_ids,
    green_mask, yellow_mask, allocs_dict, save_dir='.')
print('OK: Dashboard rendered')

# =====================================================================
# 7D: Deep dive plot
# =====================================================================
print('=== 7D: Deep dive plot ===')
test_JSAC.plot_deep_dive(
    all_metrics, para, group_ids, green_mask, yellow_mask,
    allocs_dict, sinr_min, save_dir='.')
print('OK: Deep dive rendered')

print('\n=== Step 7 smoke tests passed ===')